# ShiftGuard-SecLM: Interactive Security Reasoning Demo

Demonstrates proactive Shift-Left contextual security reasoning over arbitrary requirements, source code, and scanner findings.

In [ ]:
import json
import torch
from tokenizers import Tokenizer
from src.training.checkpointing import load_checkpoint
from src.model.security_context import SecurityContextPayload

# 1. Load Tokenizer & Model
tokenizer = Tokenizer.from_file("datasets/processed/tokenizer/tokenizer.json")
model, config, _ = load_checkpoint("/kaggle/working/checkpoints/pilot_110m/step_0000100", device="cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print("ShiftGuard-SecLM successfully loaded!")

In [ ]:
# 2. Interactive Analysis Prompt
payload = SecurityContextPayload(
    task="security_plan",
    requirement="Implement customer billing payment processor in FastAPI",
    prompt="Write a payment charge route accepting credit card token and amount",
    language="python",
    framework="fastapi",
    code="@app.post('/pay')\ndef pay(card: str, amt: float):\n    return gateway.charge(card, amt)",
)

prompt_str, _ = payload.serialize_full_sequence()
input_ids = torch.tensor([tokenizer.encode(prompt_str).ids], device=model.tok_embeddings.weight.device)

with torch.no_grad():
    generated_ids = model.generate(input_ids, max_new_tokens=256, temperature=0.2)

output_text = tokenizer.decode(generated_ids[0].tolist())
print("=== SHIFTGUARD-SECLM STRUCTURED REASONING ===")
print(output_text)